In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final_model_dataset.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Date"],
)

df.shape

(1181, 42)

In [2]:
numeric_features = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "vol_3m",
    "vol_6m",
    "vol_12m",
    "drawdown_12m",

    "spy_mom_1m",
    "spy_mom_3m",
    "spy_mom_6m",
    "spy_mom_12m",
    "spy_vol_3m",
    "spy_vol_6m",
    "spy_vol_12m",
    "spy_drawdown_12m",

    "DGS10",
    "DGS2",
    "DFF",
    "VIXCLS",
    "yield_spread_10y_2y",
    "DGS10_change_1m",
    "DGS2_change_1m",
    "DFF_change_1m",
    "VIXCLS_change_1m",
    "yield_spread_10y_2y_change_1m",
]

categorical_features = ["Ticker"]

target = "target"

In [3]:
train = df[
    (df["Date"] >= "2016-01-01")
    & (df["Date"] <= "2020-12-31")
].copy()

validation = df[
    (df["Date"] >= "2021-01-01")
    & (df["Date"] <= "2022-12-31")
].copy()

test = df[
    (df["Date"] >= "2023-01-01")
    & (df["Date"] <= "2025-11-30")
].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print()
print("Positive rates:")
print("Train:", train[target].mean())
print("Validation:", validation[target].mean())
print("Test:", test[target].mean())

Train: (591, 42)
Validation: (240, 42)
Test: (350, 42)

Positive rates:
Train: 0.45516074450084604
Validation: 0.5291666666666667
Test: 0.42


In [4]:
X_train = train[numeric_features + categorical_features]
y_train = train[target]

X_val = validation[numeric_features + categorical_features]
y_val = validation[target]

X_test = test[numeric_features + categorical_features]
y_test = test[target]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(591, 27) (591,)
(240, 27) (240,)
(350, 27) (350,)


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "ticker",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
    ]
)

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print("Training matrix:", X_train_processed.shape)
print("Validation matrix:", X_val_processed.shape)

Training matrix: (591, 35)
Validation matrix: (240, 35)


In [7]:
feature_names = preprocessor.get_feature_names_out()

print("Number of model features:", len(feature_names))
print()
print(feature_names)

Number of model features: 35

['numeric__mom_1m' 'numeric__mom_3m' 'numeric__mom_6m' 'numeric__mom_12m'
 'numeric__vol_3m' 'numeric__vol_6m' 'numeric__vol_12m'
 'numeric__drawdown_12m' 'numeric__spy_mom_1m' 'numeric__spy_mom_3m'
 'numeric__spy_mom_6m' 'numeric__spy_mom_12m' 'numeric__spy_vol_3m'
 'numeric__spy_vol_6m' 'numeric__spy_vol_12m' 'numeric__spy_drawdown_12m'
 'numeric__DGS10' 'numeric__DGS2' 'numeric__DFF' 'numeric__VIXCLS'
 'numeric__yield_spread_10y_2y' 'numeric__DGS10_change_1m'
 'numeric__DGS2_change_1m' 'numeric__DFF_change_1m'
 'numeric__VIXCLS_change_1m' 'numeric__yield_spread_10y_2y_change_1m'
 'ticker__Ticker_XLE' 'ticker__Ticker_XLF' 'ticker__Ticker_XLI'
 'ticker__Ticker_XLK' 'ticker__Ticker_XLP' 'ticker__Ticker_XLRE'
 'ticker__Ticker_XLU' 'ticker__Ticker_XLV' 'ticker__Ticker_XLY']


In [8]:
# Baseline model: Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

logistic_model.fit(
    X_train_processed,
    y_train,
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [9]:
val_predictions = logistic_model.predict(
    X_val_processed
)

val_probabilities = logistic_model.predict_proba(
    X_val_processed
)[:, 1]

In [10]:
print(
    "Accuracy:",
    accuracy_score(y_val, val_predictions),
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_val,
        val_predictions,
    ),
)

print(
    "Precision:",
    precision_score(
        y_val,
        val_predictions,
        zero_division=0,
    ),
)

print(
    "Recall:",
    recall_score(
        y_val,
        val_predictions,
        zero_division=0,
    ),
)

print(
    "F1:",
    f1_score(
        y_val,
        val_predictions,
        zero_division=0,
    ),
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val,
        val_probabilities,
    ),
)

print()
print("Confusion matrix:")
print(
    confusion_matrix(
        y_val,
        val_predictions,
    )
)

Accuracy: 0.45416666666666666
Balanced accuracy: 0.46327782036095044
Precision: 0.47560975609756095
Recall: 0.30708661417322836
F1: 0.37320574162679426
ROC-AUC: 0.47780642463939793

Confusion matrix:
[[70 43]
 [88 39]]


In [11]:
majority_class = y_train.mode()[0]

baseline_predictions = pd.Series(
    majority_class,
    index=y_val.index,
)

print(
    "Majority-class accuracy:",
    accuracy_score(
        y_val,
        baseline_predictions,
    ),
)

Majority-class accuracy: 0.4708333333333333


In [12]:
# Second model: Balanced Logistic Regression
balanced_logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

balanced_logistic_model.fit(
    X_train_processed,
    y_train,
)

balanced_val_predictions = (
    balanced_logistic_model.predict(
        X_val_processed
    )
)

balanced_val_probabilities = (
    balanced_logistic_model.predict_proba(
        X_val_processed
    )[:, 1]
)

In [13]:
print(
    "Accuracy:",
    accuracy_score(
        y_val,
        balanced_val_predictions,
    ),
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_val,
        balanced_val_predictions,
    ),
)

print(
    "Precision:",
    precision_score(
        y_val,
        balanced_val_predictions,
        zero_division=0,
    ),
)

print(
    "Recall:",
    recall_score(
        y_val,
        balanced_val_predictions,
        zero_division=0,
    ),
)

print(
    "F1:",
    f1_score(
        y_val,
        balanced_val_predictions,
        zero_division=0,
    ),
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val,
        balanced_val_probabilities,
    ),
)

print()
print("Confusion matrix:")
print(
    confusion_matrix(
        y_val,
        balanced_val_predictions,
    )
)

Accuracy: 0.4666666666666667
Balanced accuracy: 0.4716744477736743
Precision: 0.494949494949495
Recall: 0.3858267716535433
F1: 0.4336283185840708
ROC-AUC: 0.47759737997352103

Confusion matrix:
[[63 50]
 [78 49]]


In [ ]:
# Regularization L2 penalty hyperparameter tuning
results = []

C_values = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100,
]

for class_weight in [None, "balanced"]:

    for C in C_values:

        model = LogisticRegression(
            C=C,
            penalty="l2",
            class_weight=class_weight,
            max_iter=2000,
            random_state=42,
        )

        model.fit(
            X_train_processed,
            y_train,
        )

        predictions = model.predict(
            X_val_processed
        )

        probabilities = model.predict_proba(
            X_val_processed
        )[:, 1]

        results.append({
            "class_weight": (
                "none"
                if class_weight is None
                else "balanced"
            ),
            "C": C,
            "accuracy": accuracy_score(
                y_val,
                predictions,
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_val,
                    predictions,
                ),
            "precision": precision_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "f1": f1_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_val,
                probabilities,
            ),
        })


results_df = pd.DataFrame(results)

results_df.sort_values(
    "roc_auc",
    ascending=False,
)

/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/

,class_weight,C,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
1,none,0.010,0.487500,0.501603,0.532258,0.259843,0.349206,0.492161
7,balanced,0.010,0.491667,0.490419,0.520000,0.511811,0.515873,0.491673
0,none,0.001,0.470833,0.500000,0.000000,0.000000,0.000000,0.487283
6,balanced,0.001,0.504167,0.503693,0.532787,0.511811,0.522088,0.487004
10,balanced,10.000,0.462500,0.469201,0.489130,0.354331,0.410959,0.478991
5,none,100.000,0.454167,0.463766,0.475000,0.299213,0.367150,0.478643
11,balanced,100.000,0.454167,0.461327,0.477778,0.338583,0.396313,0.478573
4,none,10.000,0.454167,0.463766,0.475000,0.299213,0.367150,0.477946
3,none,1.000,0.454167,0.463278,0.475610,0.307087,0.373206,0.477806
9,balanced,1.000,0.466667,0.471674,0.494949,0.385827,0.433628,0.477597


In [ ]:
# Regularization L1 penalty hyperparameter tuning
l1_results = []

for class_weight in [None, "balanced"]:

    for C in C_values:

        model = LogisticRegression(
            C=C,
            penalty="l1",
            solver="liblinear",
            class_weight=class_weight,
            max_iter=2000,
            random_state=42,
        )

        model.fit(
            X_train_processed,
            y_train,
        )

        predictions = model.predict(
            X_val_processed
        )

        probabilities = model.predict_proba(
            X_val_processed
        )[:, 1]

        nonzero_coefficients = (
            model.coef_[0] != 0
        ).sum()

        l1_results.append({
            "class_weight": (
                "none"
                if class_weight is None
                else "balanced"
            ),
            "C": C,
            "nonzero_features":
                nonzero_coefficients,
            "accuracy": accuracy_score(
                y_val,
                predictions,
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_val,
                    predictions,
                ),
            "precision": precision_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "f1": f1_score(
                y_val,
                predictions,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_val,
                probabilities,
            ),
        })


l1_results_df = pd.DataFrame(
    l1_results
)

l1_results_df.sort_values(
    "roc_auc",
    ascending=False,
)

/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' 

,class_weight,C,nonzero_features,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,none,0.001,0,0.470833,0.500000,0.000000,0.000000,0.000000,0.500000
1,none,0.010,0,0.470833,0.500000,0.000000,0.000000,0.000000,0.500000
6,balanced,0.001,0,0.470833,0.500000,0.000000,0.000000,0.000000,0.500000
7,balanced,0.010,0,0.470833,0.500000,0.000000,0.000000,0.000000,0.500000
11,balanced,100.000,33,0.458333,0.465264,0.483516,0.346457,0.403670,0.478643
5,none,100.000,33,0.454167,0.463766,0.475000,0.299213,0.367150,0.477388
10,balanced,10.000,33,0.462500,0.469201,0.489130,0.354331,0.410959,0.476552
4,none,10.000,33,0.454167,0.463766,0.475000,0.299213,0.367150,0.476204
9,balanced,1.000,28,0.479167,0.485437,0.510638,0.377953,0.434389,0.471047
3,none,1.000,29,0.445833,0.457355,0.458333,0.259843,0.331658,0.469793


In [16]:
sparse_l1_model = LogisticRegression(
    C=0.1,
    penalty="l1",
    solver="liblinear",
    class_weight=None,
    max_iter=2000,
    random_state=42,
)

sparse_l1_model.fit(
    X_train_processed,
    y_train,
)

coefficients = pd.Series(
    sparse_l1_model.coef_[0],
    index=feature_names,
)

selected_features = (
    coefficients[coefficients != 0]
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False,
    )
)

selected_features

/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


ticker__Ticker_XLK              0.060684
numeric__mom_1m                -0.049143
numeric__yield_spread_10y_2y    0.022245
numeric__VIXCLS_change_1m       0.002585
dtype: float64

In [17]:
# Apply cross-validation with time-based folds
from sklearn.pipeline import Pipeline

validation_years = [2019, 2020, 2021, 2022]

for validation_year in validation_years:
    fold_train = df[
        (df["Date"].dt.year >= 2016)
        & (df["Date"].dt.year < validation_year)
    ]

    fold_val = df[
        df["Date"].dt.year == validation_year
    ]

    print(
        validation_year,
        "train:",
        len(fold_train),
        "validation:",
        len(fold_val),
    )

2019 train: 351 validation: 120
2020 train: 471 validation: 120
2021 train: 591 validation: 120
2022 train: 711 validation: 120


In [18]:
candidate_models = []

C_values = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100,
]

for penalty in ["l1", "l2"]:
    for class_weight in [None, "balanced"]:
        for C in C_values:
            candidate_models.append({
                "penalty": penalty,
                "class_weight": class_weight,
                "C": C,
            })

len(candidate_models)

24

In [19]:
cv_results = []

for config in candidate_models:

    for validation_year in validation_years:

        fold_train = df[
            (df["Date"].dt.year >= 2016)
            & (df["Date"].dt.year < validation_year)
        ].copy()

        fold_val = df[
            df["Date"].dt.year == validation_year
        ].copy()

        X_fold_train = fold_train[
            numeric_features + categorical_features
        ]

        y_fold_train = fold_train[target]

        X_fold_val = fold_val[
            numeric_features + categorical_features
        ]

        y_fold_val = fold_val[target]

        # New preprocessing object for EVERY fold
        fold_preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    numeric_features,
                ),
                (
                    "ticker",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore",
                    ),
                    categorical_features,
                ),
            ]
        )

        model = LogisticRegression(
            penalty=config["penalty"],
            C=config["C"],
            class_weight=config["class_weight"],
            solver="liblinear",
            max_iter=2000,
            random_state=42,
        )

        pipeline = Pipeline([
            ("preprocessor", fold_preprocessor),
            ("model", model),
        ])

        pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        predictions = pipeline.predict(
            X_fold_val
        )

        probabilities = pipeline.predict_proba(
            X_fold_val
        )[:, 1]

        cv_results.append({
            "validation_year": validation_year,
            "penalty": config["penalty"],
            "class_weight": (
                "none"
                if config["class_weight"] is None
                else "balanced"
            ),
            "C": config["C"],
            "roc_auc": roc_auc_score(
                y_fold_val,
                probabilities,
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_fold_val,
                    predictions,
                ),
            "f1": f1_score(
                y_fold_val,
                predictions,
                zero_division=0,
            ),
        })

/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' 

In [20]:
cv_results_df = pd.DataFrame(cv_results)

cv_summary = (
    cv_results_df
    .groupby(
        ["penalty", "class_weight", "C"],
        as_index=False,
    )
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        mean_f1=("f1", "mean"),
    )
    .sort_values(
        "mean_roc_auc",
        ascending=False,
    )
)

cv_summary.head(10)

,penalty,class_weight,C,mean_roc_auc,std_roc_auc,mean_balanced_accuracy,mean_f1
0,l1,balanced,0.001,0.500000,0.000000,0.500000,0.000000
6,l1,none,0.001,0.500000,0.000000,0.500000,0.000000
7,l1,none,0.010,0.500000,0.000000,0.500000,0.000000
1,l1,balanced,0.010,0.500000,0.000000,0.500000,0.000000
15,l2,balanced,1.000,0.499692,0.043640,0.486420,0.350465
21,l2,none,1.000,0.498994,0.043465,0.503352,0.316949
3,l1,balanced,1.000,0.498128,0.054102,0.488567,0.439174
16,l2,balanced,10.000,0.498048,0.036516,0.479562,0.298478
9,l1,none,1.000,0.497707,0.054767,0.482252,0.376242
22,l2,none,10.000,0.497700,0.035974,0.492034,0.266310


In [21]:
baseline_yearly_results = cv_results_df[
    (cv_results_df["penalty"] == "l2")
    & (cv_results_df["class_weight"] == "balanced")
    & (cv_results_df["C"] == 1)
]

baseline_yearly_results

,validation_year,penalty,class_weight,C,roc_auc,balanced_accuracy,f1
84,2019,l2,balanced,1.0,0.542484,0.537084,0.200000
85,2020,l2,balanced,1.0,0.500000,0.489899,0.411765
86,2021,l2,balanced,1.0,0.516569,0.478697,0.415094
87,2022,l2,balanced,1.0,0.439714,0.440000,0.375000


### Baseline result

Expanding-window validation showed that logistic regression using the
original-style 35-feature set produced mean ROC-AUC near 0.50 across
2019–2022 validation periods.

Strong L1 regularization produced degenerate models with all coefficients
shrunk to zero. Among non-degenerate specifications, neither L1 nor L2
regularization generated stable improvement above chance.

This suggests that the original feature representation contains limited
stable linear predictive signal for next-month sector outperformance.